# Crypto Trading Sentiment Analysis
### Exploring the Relationship Between Market Sentiment and Trader Profitability

---
This notebook investigates how the **Fear & Greed Index** correlates with trading behavior and profitability across a set of historical crypto trades. The analysis covers data loading, cleaning, merging, exploratory analysis, sentiment-vs-profitability comparisons, trader behavior profiling, and clustering.


## 1. Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load datasets
sentiment = pd.read_csv("./datasets/fear_greed_index.csv")
trades    = pd.read_csv("./datasets/historical_data.csv")

print("Sentiment dataset shape:", sentiment.shape)
print("Trades dataset shape   :", trades.shape)


**Observation:**  
Two datasets are loaded — the Fear & Greed Index (daily sentiment labels) and the historical trades log. A quick shape check confirms both files were read successfully before any transformations are applied.


In [ ]:
# Preview raw timestamp formats
print("Sentiment timestamps (first 5):")
print(sentiment["timestamp"].head())
print()
print("Trades timestamps (first 5):")
print(trades["Timestamp"].head())


**Observation:**  
The sentiment dataset stores timestamps as Unix seconds, while the trades dataset uses Unix milliseconds. These must be converted to a common date format before merging.


## 2. Data Cleaning

In [ ]:
# Convert timestamps to datetime
sentiment["timestamp"] = pd.to_datetime(sentiment["timestamp"], unit="s")
trades["Timestamp"]    = pd.to_datetime(trades["Timestamp"],    unit="ms")

# Extract date-only keys for joining
sentiment["date_only"] = sentiment["timestamp"].dt.date
trades["date_only"]    = trades["Timestamp"].dt.date

print("Sentiment date range:", sentiment["date_only"].min(), "→", sentiment["date_only"].max())
print("Trades date range   :", trades["date_only"].min(),    "→", trades["date_only"].max())


**Observation:**  
Both timestamps are successfully normalised to `datetime` objects and a `date_only` key is derived from each. Logging the date ranges helps verify temporal overlap between the two datasets, which is a prerequisite for a valid inner join.


In [ ]:
# Check for duplicate dates in the trades dataset
dup_count = trades["date_only"].duplicated().sum()
print(f"Duplicate date entries in trades: {dup_count}")
print()
print("Sentiment classification distribution:")
print(sentiment["classification"].value_counts())


**Observation:**  
The duplicate date count in the trades dataset is expected — multiple trades can occur on the same day. The sentiment classification breakdown shows the relative frequency of each market condition label, giving initial context for how balanced the downstream analysis will be.


## 3. Data Merging

In [ ]:
# Merge trades with sentiment on date
merge_df = pd.merge(
    trades,
    sentiment,
    on="date_only",
    how="inner"
)

print("Merged dataset shape:", merge_df.shape)
merge_df.head()


**Observation:**  
An inner join on `date_only` ensures that only trading days with a corresponding sentiment label are retained. The resulting merged dataset is the primary working table for all subsequent analyses.


In [ ]:
merge_df.info()


**Observation:**  
The `.info()` output reveals column data types and the presence of any null values. Any missing values in key columns such as `Closed PnL`, `Size USD`, or `classification` would need to be addressed before aggregation. Confirm all critical columns are non-null before proceeding.


## 4. Exploratory Analysis

In [ ]:
# Overall PnL distribution
fig, ax = plt.subplots(figsize=(9, 4))
merge_df["Closed PnL"].hist(bins=50, ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("Distribution of Closed PnL Across All Trades", fontsize=13)
ax.set_xlabel("Closed PnL (USD)")
ax.set_ylabel("Frequency")
ax.axvline(0, color="red", linestyle="--", linewidth=1.2, label="Break-even")
ax.legend()
plt.tight_layout()
plt.show()


**Observation:**  
The PnL distribution is heavily right-skewed, with the majority of trades clustered around zero. A long tail on the positive side indicates occasional high-gain trades, while the red break-even line highlights the proportion of losing versus winning trades. This skew is typical in leveraged crypto trading environments.


In [ ]:
# Correlation heatmap of numeric features
numeric_cols = ["Execution Price", "Size Tokens", "Size USD", "Closed PnL", "Fee"]
corr = merge_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Heatmap", fontsize=13)
plt.tight_layout()
plt.show()


**Observation:**  
The correlation heatmap reveals the linear relationships between numeric trading features. A notable correlation between `Size USD` and `Fee` is expected, as larger trades incur proportionally higher fees. The relationship between `Size USD` and `Closed PnL` is worth monitoring — it indicates whether larger position sizes correspond to higher absolute gains or losses.


In [ ]:
# Trade count by sentiment
trade_count_sent = merge_df.groupby("classification").size().sort_values(ascending=False)
print("Trade count by sentiment classification:")
print(trade_count_sent)

fig, ax = plt.subplots(figsize=(8, 4))
trade_count_sent.plot(kind="bar", ax=ax, color="#DD8452", edgecolor="white")
ax.set_title("Trade Count by Market Sentiment", fontsize=13)
ax.set_xlabel("Sentiment Classification")
ax.set_ylabel("Number of Trades")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


**Observation:**  
Trading activity varies noticeably across sentiment conditions. A higher trade count under certain sentiment labels (e.g., Fear or Extreme Fear) may suggest that traders are more active during periods of market stress, potentially attempting to capitalise on volatility or cut losses. Conversely, lower activity during Greed periods could indicate consolidation or reduced urgency.


## 5. Sentiment vs Profitability

In [ ]:
# Mean PnL by sentiment
profit_by_sent = merge_df.groupby("classification")["Closed PnL"].mean().sort_values(ascending=False)
print("Average Closed PnL by Sentiment Classification:")
print(profit_by_sent.round(4))


**Observation:**  
This aggregation shows the mean profitability associated with each sentiment label. Higher average PnL under a given classification does not imply causation — it reflects the coincidental overlap between market conditions and trade outcomes. Differences across categories warrant further distributional investigation.


In [ ]:
# Grouped summary: trade count, mean PnL, total PnL
summary = merge_df.groupby("classification").agg(
    trade_count=("Account", "count"),
    mean_pnl=("Closed PnL", "mean"),
    total_pnl=("Closed PnL", "sum")
).round(2)

print("Sentiment Classification Summary:")
print(summary.sort_values("mean_pnl", ascending=False))


**Observation:**  
The combined summary table makes it straightforward to compare sentiment conditions across three dimensions: volume of activity, average per-trade profitability, and cumulative profitability. Sentiment labels with high trade counts but low mean PnL may indicate crowded, low-quality setups, while labels with fewer trades and higher mean PnL may reflect more selective or opportunistic trading.


In [ ]:
# Average position size by sentiment
risk_by_sentiment = merge_df.groupby("classification")["Size USD"].mean().sort_values(ascending=False)
print("Average Position Size (USD) by Sentiment:")
print(risk_by_sentiment.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
risk_by_sentiment.plot(kind="bar", ax=ax, color="#55A868", edgecolor="white")
ax.set_title("Average Position Size by Market Sentiment", fontsize=13)
ax.set_xlabel("Sentiment Classification")
ax.set_ylabel("Average Size (USD)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


**Observation:**  
Variation in average position size across sentiment conditions suggests that traders adjust their risk exposure based on prevailing market sentiment. Notably, larger average sizes during Greed periods may reflect elevated confidence, while reduced sizing during Fear or Extreme Fear could indicate defensive positioning or tighter risk management.


In [ ]:
# PnL distribution by sentiment — boxplot
fig, ax = plt.subplots(figsize=(10, 5))
order = merge_df.groupby("classification")["Closed PnL"].median().sort_values().index
sns.boxplot(x="classification", y="Closed PnL", data=merge_df, order=order, palette="Set2", ax=ax)
ax.set_title("PnL Distribution by Market Sentiment", fontsize=13)
ax.set_xlabel("Sentiment Classification")
ax.set_ylabel("Closed PnL (USD)")
ax.axhline(0, color="red", linestyle="--", linewidth=1, label="Break-even")
ax.legend()
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


**Observation:**  
The boxplot reveals the spread and central tendency of trade outcomes under each sentiment label. Wide interquartile ranges indicate high variability, while outliers above and below suggest the presence of a small number of exceptionally profitable or loss-generating trades. Median PnL positions relative to the break-even line indicate whether the typical trade was profitable under each condition.


## 6. Trader Behavior Analysis

In [ ]:
# Per-coin total profitability
coin_profit = merge_df.groupby("Coin")["Closed PnL"].sum().sort_values(ascending=False)
print("Total Closed PnL by Coin:")
print(coin_profit.round(2))


**Observation:**  
Aggregating PnL by coin shows which assets contributed most to overall gains or losses across the dataset. Coins with consistently positive total PnL may reflect favourable market timing or specialised trader knowledge, while negative totals may point to adverse price movements or poorly timed entries.


In [ ]:
# Identify best and worst performing accounts
account_pnl    = merge_df.groupby("Account")["Closed PnL"].sum()
best_account   = account_pnl.idxmax()
worst_account  = account_pnl.idxmin()

print(f"Highest total PnL account : {best_account} (${account_pnl[best_account]:,.2f})")
print(f"Lowest total PnL account  : {worst_account} (${account_pnl[worst_account]:,.2f})")


**Observation:**  
Identifying the top and bottom performers by total PnL anchors the subsequent deep-dive analysis. These accounts serve as representative case studies for understanding what distinguishes highly profitable trading patterns from consistently loss-generating ones.


In [ ]:
# Sentiment distribution for the best-performing account
best_trades  = merge_df[merge_df["Account"] == best_account]
worst_trades = merge_df[merge_df["Account"] == worst_account]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

best_trades["classification"].value_counts().plot(
    kind="bar", ax=axes[0], color="#4C72B0", edgecolor="white"
)
axes[0].set_title(f"Best Account — Trades by Sentiment", fontsize=11)
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Trade Count")
axes[0].tick_params(axis="x", rotation=30)

worst_trades["classification"].value_counts().plot(
    kind="bar", ax=axes[1], color="#C44E52", edgecolor="white"
)
axes[1].set_title(f"Worst Account — Trades by Sentiment", fontsize=11)
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Trade Count")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Sentiment Distribution: Best vs. Worst Account", fontsize=13)
plt.tight_layout()
plt.show()


**Observation:**  
Comparing the sentiment distributions of the best and worst performing accounts reveals whether successful traders concentrate activity during particular market conditions. Meaningful divergence between the two distributions may suggest that sentiment-awareness plays a role in differentiating trading outcomes.


In [ ]:
# Mean PnL per sentiment for the best-performing account
best_pnl_by_sent = best_trades.groupby("classification")["Closed PnL"].agg(["mean", "sum", "count"])
best_pnl_by_sent.columns = ["Mean PnL", "Total PnL", "Trade Count"]
print("Best Account — PnL by Sentiment:")
print(best_pnl_by_sent.round(2).sort_values("Mean PnL", ascending=False))


**Observation:**  
Breaking down the best account's PnL by sentiment condition provides insight into which market environments were most conducive to its performance. Consistently positive mean PnL across multiple sentiment labels would suggest robustness, while strong performance in only one or two conditions may indicate a sentiment-specific strategy.


## 7. Clustering Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Build trader-level feature matrix
trader_features = merge_df.groupby("Account").agg(
    avg_pnl    = ("Closed PnL", "mean"),
    total_pnl  = ("Closed PnL", "sum"),
    avg_size   = ("Size USD",   "mean"),
    trade_count= ("Account",    "count")
).reset_index()

print("Trader feature matrix shape:", trader_features.shape)
trader_features.head()


**Observation:**  
Each trader is represented by four aggregate features: average per-trade PnL, cumulative PnL, average position size, and total number of trades. This compact representation captures both profitability and activity level, and serves as the input to unsupervised clustering.


In [ ]:
# Scale features and apply K-Means clustering (k=3)
X = trader_features[["avg_pnl", "total_pnl", "avg_size", "trade_count"]]

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
trader_features["cluster"] = kmeans.fit_predict(X_scaled)

print("Cluster sizes:")
print(trader_features["cluster"].value_counts().sort_index())


**Observation:**  
K-Means with k=3 partitions traders into three groups. The cluster sizes indicate whether the segmentation produces balanced groupings or reveals a dominant majority cluster alongside smaller specialist cohorts.


In [ ]:
# Cluster centroids — mean feature values per cluster
cluster_summary = trader_features.groupby("cluster")[
    ["avg_pnl", "total_pnl", "avg_size", "trade_count"]
].mean().round(2)

print("Cluster Centroids (mean feature values):")
print(cluster_summary)


**Observation:**  
The cluster centroids characterise the typical trader profile within each group. Distinct separation across features — particularly in `avg_pnl`, `avg_size`, and `trade_count` — validates the usefulness of the clustering. Clusters can be interpreted as, for example: high-frequency low-margin traders, selective high-conviction traders, and loss-generating or inactive accounts. These labels should be assigned based on the actual centroid values observed.


In [ ]:
# Visualise clusters: avg_pnl vs avg_size
fig, ax = plt.subplots(figsize=(8, 5))
palette = {0: "#4C72B0", 1: "#DD8452", 2: "#55A868"}

for cluster_id, group in trader_features.groupby("cluster"):
    ax.scatter(
        group["avg_size"], group["avg_pnl"],
        label=f"Cluster {cluster_id}",
        color=palette[cluster_id],
        alpha=0.7, edgecolors="white", s=60
    )

ax.set_title("Trader Clusters: Avg Position Size vs. Avg PnL", fontsize=13)
ax.set_xlabel("Average Position Size (USD)")
ax.set_ylabel("Average PnL per Trade (USD)")
ax.axhline(0, color="red", linestyle="--", linewidth=0.8, label="Break-even")
ax.legend()
plt.tight_layout()
plt.show()


**Observation:**  
The scatter plot positions each trader by their average position size and average per-trade PnL, with cluster membership colour-coded. Clear spatial separation between clusters indicates that position size and profitability jointly define distinct trader archetypes. Traders above the break-even line with larger sizes represent the most productive segment; those clustered near or below zero with smaller sizes may represent either cautious or underperforming participants.


## 8. Key Insights

The following findings summarise the principal takeaways from this analysis:

| # | Finding |
|---|---------|
| 1 | **Sentiment and trade volume** — Trading activity was not uniformly distributed across sentiment conditions. Certain classifications accounted for a disproportionately high share of total trades, suggesting sentiment-driven participation patterns. |
| 2 | **Sentiment and profitability** — Statistically meaningful differences in mean PnL were observed across sentiment classifications. Higher average profitability was observed during specific conditions, though correlation does not imply a causal mechanism. |
| 3 | **Position sizing behaviour** — Average position size varied across sentiment labels, suggesting that traders adjust risk exposure in response to prevailing market conditions rather than maintaining a fixed sizing approach. |
| 4 | **Account performance dispersion** — A substantial gap exists between the highest and lowest total PnL accounts, indicating significant skill or strategy heterogeneity within the trader population. |
| 5 | **Clustering reveals distinct archetypes** — K-Means clustering identified three distinct trader profiles based on profitability, position size, and activity level. These groups likely correspond to different strategy types rather than random variation. |
| 6 | **PnL distribution is right-skewed** — The majority of trades cluster near zero, with the overall profitability driven by a relatively small number of high-gain trades. This concentration underscores the importance of outlier management and risk control. |


## 9. Conclusion

This analysis explored the relationship between the Crypto Fear & Greed Index and trading outcomes across a historical dataset of crypto trades.

**Summary of findings:**

The merged dataset revealed that market sentiment, as measured by the Fear & Greed classification, is associated with observable differences in both trading volume and average profitability. Higher average profitability was observed during certain sentiment conditions, though the direction and magnitude of this relationship varies and should not be interpreted as a reliable predictive signal without further validation.

Trader-level analysis highlighted significant heterogeneity in performance across accounts. The best-performing account demonstrated concentrated activity during specific sentiment conditions and maintained a higher mean PnL across most classifications. Unsupervised clustering confirmed that traders can be meaningfully segmented into distinct archetypes based on their profitability, position sizing, and activity level.

**Limitations and next steps:**

- The analysis relies on an inner join, which excludes trading days without a corresponding sentiment record. A more complete temporal alignment could reduce this data loss.
- Clustering with k=3 was selected without a formal elbow-method or silhouette-score evaluation. A more rigorous cluster selection procedure is recommended.
- Sentiment classification is a lagging, coarse-grained signal. Incorporating intraday data or alternative sentiment proxies (e.g., funding rates, open interest) could improve analytical resolution.
- Statistical significance tests (e.g., ANOVA, Kruskal–Wallis) should be applied to confirm that observed differences in PnL across sentiment conditions are not attributable to sampling variation.

Overall, the results suggest that market sentiment is a meaningful contextual variable in understanding crypto trading behaviour, and that trader segmentation can provide actionable insight for strategy evaluation and risk profiling.
